# 하이퍼파라미터 자동 탐색 (run_hpo)

`run_experiment_augmented.ipynb`에서 만든 것과 **동일한**(누수 안전 그룹화 적용)
`train_aug`/`val_aug`를 그대로 재사용해서, `src/hpo.py`의 Optuna 베이지안 탐색으로
모델별 하이퍼파라미터(lr/weight_decay/batch_size 등)를 자동으로 탐색한다.

핵심 아이디어:
1. **탐색 단계는 epoch을 줄여서(`hpo_epochs`) 빠르게 "방향"만 잡는다.** 전체
   epoch(30)으로 매 trial을 다 돌리면 너무 오래 걸린다 — 예를 들어 YOLO는
   8epoch, torchvision 계열은 5epoch 정도로 줄여서 상대적 우열만 비교한다.
2. **각 trial은 `experiments/hpo/<exp_name>_trial{N}/`에 별도로 저장**되므로
   기존 `metrics.json` 기반 비교 코드를 그대로 쓸 수 있다.
3. **가장 좋았던 조합을 찾으면, 그 하이퍼파라미터로 전체 epoch(30) config를
   새로 만들어 최종 재학습**한다(탐색용 짧은 학습 결과를 그대로 "최종 성능"으로
   믿지 않는다 — 방향만 참고).

의존성: `optuna`가 필요하다(`pyproject.toml`에 추가해 뒀지만, 다른 팀원들과 같이
쓰는 monorepo 루트 `pyproject.toml`이라 실제 설치는 `uv sync`를 각자 실행해야 한다).

In [1]:
import os

# RF-DETR(Multi-Scale Deformable Attention)의 grid_sample backward가 PyTorch
# MPS 백엔드에 아직 없어서(forward만 지원) 이 연산만 CPU로 자동 폴백하게 켜둔다.
# torch를 어디서든 import하기 전에 설정해야 가장 안전하므로 첫 셀 맨 위에 둔다.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

from utils.sys_utils import add_experiment_root

PROJECT_ROOT = add_experiment_root()
print("PROJECT_ROOT:", PROJECT_ROOT)


PROJECT_ROOT: /Users/joelchoi/study/projects/project1_3team/beamsearch/CJY


In [2]:
from pathlib import Path

from src.audit_leakage import audit
from src.data.aihub_converter import convert_aihub_to_coco
from src.data.coco_to_yolo import prepare_yolo_dataset
from src.data.kaggle_converter import convert_kaggle_to_coco
from src.data.merge import merge_for_augmentation
from src.data.subset import build_leakage_safe_groups, create_subset, exclude_images
from src.hpo import run_hpo, summarize_study
from src.utils import load_config

CONFIGS_DIR = PROJECT_ROOT / "configs" / "experiment"
EXPERIMENTS_DIR = PROJECT_ROOT / "experiments"
HPO_DIR = (
    EXPERIMENTS_DIR / "hpo"
)  # trial들은 별도 하위 폴더에 모아서 본 실험과 섞이지 않게
DATA_DIR = PROJECT_ROOT / "data"
YOLO_DIR = (
    DATA_DIR / "yolo_aug_clean"
)  # run_experiment_augmented.ipynb와 동일 경로(있으면 재사용)
AIHUB_COMBOS = [1, 3, 4, 5, 6]


## 1단계 — 데이터 준비 (run_experiment_augmented.ipynb와 동일)

이미 그 노트북을 돌려서 `data/yolo_aug_clean/`이 만들어져 있다면 아래 변환 과정을
그대로 다시 실행해도 되고(결정적이라 매번 같은 결과), 시간을 아끼고 싶으면 이
셀들을 스킵하고 `train_aug`/`val_aug`만 다시 만드는 부분부터 시작해도 된다.

In [ ]:
kaggle_coco = convert_kaggle_to_coco()
allowed_ids = {c["id"] for c in kaggle_coco["categories"]}
aihub_coco_raw = convert_aihub_to_coco(combo_nums=AIHUB_COMBOS)


Kaggle 학습 데이터 변환 완료: 이미지 232장, 어노테이션 763개, 클래스 56개
변환 중: TS_01/TL_01 ...
  → 이미지 1188장, 어노테이션 4738개
변환 중: TS_03/TL_03 ...
  → 이미지 1491장, 어노테이션 5676개
변환 중: TS_04/TL_04 ...
경고: ._K-005094-005886-012778-037777_0_2_0_2_90_000_200.json 파일을 읽을 수 없어 건너뜁니다.
  → 이미지 1392장, 어노테이션 5023개
변환 중: TS_05/TL_05 ...
  → 이미지 462장, 어노테이션 1845개
변환 중: TS_06/TL_06 ...
경고: K-012247-016551-033009-044199_0_2_0_2_90_000_200.json 파일을 읽을 수 없어 건너뜁니다.
  → 이미지 1122장, 어노테이션 4466개

총 이미지: 5655, 어노테이션: 21748, 클래스: 91


## 2단계 — YOLO 하이퍼파라미터 탐색

`exp016_yolo11n_aug_clean.yaml`을 기준(base_config)으로 `src/hpo.DEFAULT_SEARCH_SPACES["yolo"]`
(lr, weight_decay, batch_size, optimizer)를 탐색한다. `hpo_epochs=8`로 짧게 돌려
방향만 잡는다 — trial 수(`n_trials`)와 `hpo_epochs`는 가진 시간에 맞춰 조정할 것.

In [ ]:
yolo_base_cfg = load_config(CONFIGS_DIR / "exp022_yolo11s_aug_clean.yaml")


## 3단계 — 가장 좋았던 조합으로 "본 실험용" config 만들기

탐색은 짧은 epoch(8/5)로 한 결과라 그 자체를 최종 성능으로 쓰면 안 된다. 가장
좋았던 하이퍼파라미터를 원래 config(전체 epoch=30)에 적용해 새 파일로 저장한
다음, `run_experiment_augmented.ipynb`처럼 `run_yolo_experiment`/`train_torchvision`으로
전체 epoch 재학습해서 최종 점수를 확인할 것.

In [ ]:
import json
import yaml


def materialize_best_config(base_cfg: dict, study, suffix: str = "hpo_best") -> Path:
    """study.best_params를 base_cfg(전체 epoch 유지)에 적용해 새 yaml로 저장."""
    best_cfg = json.loads(json.dumps(base_cfg))  # 깊은 복사
    for dotted_key, value in study.best_params.items():
        keys = dotted_key.split(".")
        node = best_cfg
        for k in keys[:-1]:
            node = node[k]
        node[keys[-1]] = value

    new_name = f"{base_cfg['experiment']['name']}_{suffix}"
    best_cfg["experiment"]["name"] = new_name
    out_path = CONFIGS_DIR / f"{new_name}.yaml"
    with open(out_path, "w", encoding="utf-8") as f:
        yaml.dump(best_cfg, f, allow_unicode=True, default_flow_style=False)
    print(
        f"저장: {out_path} (mAP@75:95 탐색 점수 {study.best_value:.4f}, epochs={best_cfg['training']['epochs']}로 복원됨)"
    )
    return out_path


yolo_best_path = materialize_best_config(yolo_base_cfg, yolo_study)

### Yolo Best Training with Hyper Parameters

In [ ]:
from src.train_yolo import run_yolo_experiment

# yaml_path = prepare_yolo_dataset(train_aug, val_aug, YOLO_DIR, symlink=True)
# CLASS_MAP = YOLO_DIR / "class_map.json"  # 전체 클래스(K-코드) 매핑

yolo_cfg = load_config(
    CONFIGS_DIR / "exp016_yolo11s_aug_clean_hpo_best.yaml"
)  # yolo11s


### 11n | kaggle + AIHub 1, 3
```json
{'map75': 0.9949999999999999, 'map75_95': 0.9916084353741497, 'model': 'yolo11n', 'epochs': 60}
```
### 11s | kaggle + AIHub 1~6(2제외)
```json
{'map75': 0.9949999999999999, 'map75_95': 0.9944285714285713, 'model': 'yolo11s', 'epochs': 60}
```
<sub>{'training.lr': 0.0002105435340076873, 'training.weight_decay': 3.058021141335009e-05, 'training.batch_size': 32, 'training.optimizer': 'SGD'}</sub>

## 8단계 — 제출용 class_map(56 제한) + 추론 + 제출 (YOLO 기준)
현재 추론 파이프라인(`src/inference.py`)은 YOLO 전용이라 exp016(그룹화 fix 버전)
가중치로 제출 파일을 만든다.

In [ ]:
from src.inference import restrict_class_map, run_inference, save_submission

CLASS_MAP = DATA_DIR / "yolo_aug_clean" / "class_map.json"  # 전체 클래스(K-코드) 매핑
EXP_NAME = yolo_cfg["experiment"]["name"]
KAGGLE_DATA = (
    Path.home()
    / ".cache/kagglehub/competitions/ai12-level1-project/sprint_ai_project1_data"
)
WEIGHTS = EXPERIMENTS_DIR / EXP_NAME / "weights" / "best.pt"
SUBMISSION = PROJECT_ROOT / "submissions" / f"{EXP_NAME}_submission.csv"

# 56 클래스만 남긴 제출용 class_map
CLASS_MAP_SUBMIT = restrict_class_map(CLASS_MAP, allowed_ids)


제출용 class_map: 56/93개 클래스 유지(나머지 -1) → /Users/joelchoi/study/projects/project1_3team/beamsearch/CJY/data/yolo_aug_clean/class_map_submit.json


## 9단계 — 제출 파일 점검

In [ ]:
import pandas as pd

sub_df = pd.read_csv(SUBMISSION)
test_ids = {int(p.stem) for p in (KAGGLE_DATA / "test_images").glob("*.png")}
pred_ids = set(sub_df["image_id"].unique())
bad = set(sub_df["category_id"].unique()) - allowed_ids
print(
    f"행 {len(sub_df)}, 예측 이미지 {len(pred_ids)}/{len(test_ids)}, "
    f"고유 category {sub_df['category_id'].nunique()}"
)
print("56 외 category_id?", bad if bad else "없음(정상)")
sub_df.head()


제출용 class_map: 56/93개 클래스 유지(나머지 -1) → /Users/joelchoi/study/projects/project1_3team/beamsearch/CJY/data/yolo_aug_clean/class_map_submit.json
추론 대상: 842장


KeyboardInterrupt: 